In [8]:

import gzip
import random

##### FASTA #####
def reading_dnaFasta(filename):
    '''
    WHAT IT DOES
    '''
    infile = open(filename, 'r')
    header = ''
    sequence = ''

    for line in infile:
        line = line.strip()
        if line.startswith('>'):
            if header != '':
                yield header, sequence
            header = line[1:]
            sequence = ''
        else:
            sequence += line
    if sequence != '':
        yield header, sequence

    infile.close()


##### Length and sequences dict #####
def read_fasta_dict(filename):
    '''
    Returns gene_length and gene_sequences with the key being the gene
    '''
    gene_length = {}
    gene_sequences = {}

    for header, seq in reading_dnaFasta(filename):
        name = header.strip().split(' ')[0]
        gene_length[name] = len(seq)
        gene_sequences[name] = seq

    return gene_length, gene_sequences


##### Reverse Complement Strand #####
def reverse_com_strand(sequence):
    '''
    Returns the reverse complement of the input sequence
    '''
    comp = {'A':'T','T':'A','G':'C','C':'G'}
    rev = []
    for base in sequence[::-1]:
        if base in comp:
            rev.append(comp[base])
        else:
            rev.append('N')

    return ''.join(rev)


##### Kmer and position of kmer in the sequence #####
def nineteen_pos(DNA):
    '''
    Takes out a DNA sequence and returns a kmer and its position in the sequence
    '''
    for i in range(len(DNA) - 18):
        kmer = DNA[i:i+19]
        if 'N' not in kmer:
            yield i, kmer


##### Dict of all kmers for all resistance genes (with their positions) #####
def create_dict_kmer(fasta_file):
    '''
    For the resistance genes. Takes all genes and their sequences.
    Computes the kmers for all gene sequences, both forward and reverse.
    Each k-mer maps to a list of (gene_name, position, strand) entries
    '''
    kmer_index = {}

    for header, sequence in reading_dnaFasta(fasta_file):
        gene_name = header.strip().split(' ')[0]

        for position, kmer in nineteen_pos(sequence):
            if kmer not in kmer_index:
                kmer_index[kmer] = []
            kmer_index[kmer].append((gene_name, position, '+'))
        
        rev = reverse_com_strand(sequence)
        for rev_position, kmer in nineteen_pos(rev):
            if kmer not in kmer_index:
                kmer_index[kmer] = []
            kmer_index[kmer].append((gene_name, rev_position, '-'))

    return kmer_index


##### FASTQ File  #####

def fastqread(filename):
    '''
    Inputs the fastq file and yields the sequence of each read in the file
    '''
    infile = gzip.open(filename, 'rt')

    while True:
        infile.readline()
        seq = infile.readline().rstrip()
        infile.readline()
        infile.readline()

        if len(seq) == 0:
            break

        yield seq

    infile.close()


# ----------------------------
# MAIN ALGORITHM
# ----------------------------


##### Matching the reads to the reference genes #######
def match_reads_to_genes(fastq_files, resistance_kmer, gene_lengths):
    '''
    Map sequencing reads to reference genes using k-mer matches (19 length kmers)
    The coverage is only counted at position supported byt at least one matching kmer
    
    For each read, we:
     - Find all possible alignments based on shared k-mers
     - Keep only the best alignments (the ones with most matches)
     - Increase coverage across the positions of the aligned region of the gene that are backed by k-mer matches

    This approach avoids false gaps caused by SNPs or sequencing errors,
    while still distinguishing between similar genes.
    '''

    # check if we can use it like this
    dict_depth = {gene: [0] * length for gene, length in gene_lengths.items()}
    unique_match_counts = {gene: 0 for gene in gene_lengths}

    for fastq in fastq_files:
        for read in fastqread(fastq):
            read_len = len(read)

            if read_len < 19:
                continue

            # First filter to remove 
            possible_hit = False
            for pos in range(0, read_len - 18, 20):
                kmer = read[pos:pos + 19]
                if 'N' in kmer:
                    continue
                if kmer in resistance_kmer:
                    possible_hit = True
                    break
            if not possible_hit:
                continue

            # Group k-mer hits per read into alignment hypotheses
            # (gene, alignment_shift, strand) -> [total_hits, unique_hits, matched_read_positions]
            reads_found_gene = {}

            for read_pos in range(read_len - 18):
                kmer_read = read[read_pos:read_pos + 19]
                if 'N' in kmer_read or kmer_read not in resistance_kmer:
                    continue

                genes_matched = resistance_kmer[kmer_read]  # matches = list of (gene, alignment_shift, strand) for all hits

                # kmer is unique if every entry maps to the same gene
                first_gene_matched = genes_matched[0][0]
                unique = True 
                for entry in genes_matched:
                    if entry[0] != first_gene_matched: # If any match has a different gene name it means that it is not unique 
                        unique = False
                        break

                for gene, gene_pos, strand in genes_matched:
                    alignment_shift = gene_pos - read_pos
                    id = (gene, alignment_shift, strand)
                    if id not in reads_found_gene:
                        # [total_hits, unique_hits]
                        reads_found_gene[id] = [0, 0]    # total hits
                    reads_found_gene[id][0] += 1

                    if unique:
                        reads_found_gene[id][1] += 1
                
            if not reads_found_gene:
                continue 

            # Best-hits attribution: only credit alignments tied at max hits
            best_hits = 0
            for info in reads_found_gene.values():
                if info[0] > best_hits:
                    best_hits = info[0]

            if best_hits < 3:  # ignore noise/ random matches
                continue

            for (gene, shift, strand), (hit_count, unique_hits) in reads_found_gene.items():
                if hit_count == best_hits:
                    gene_len = gene_lengths[gene]
                    # Convert each read position into a gene position
                    if strand == '+':
                        start = shift
                        end = shift + read_len - 1
                    else:
                        start = gene_len - shift - read_len  # map it in the forward coordinates
                        end = gene_len - 1 - shift
                    
                    if start < 0:
                        start = 0
                    if end > gene_len - 1:
                        end = gene_len - 1

                    for p in range(start, end + 1):
                        dict_depth[gene][p] += 1
                    unique_match_counts[gene] += unique_hits
    
    return dict_depth, unique_match_counts



##### Compute coverage and depth; Posterior filtering #######

def calculate_coverage_depth(dict_depth, gene_length, min_cov=95, min_depth=10, pos_min_depth=1):
    '''
    Filter genes by minimum core coverage and average core depth
    '''
    filtered_genes = {}

    for gene, pos_counts in dict_depth.items():   
        gene_len = gene_length[gene]

        core_start = 18
        core_end = gene_len - 18

        if core_end <= core_start:
            continue

        core = pos_counts[core_start:core_end]
        n = len(core)

        covered = sum(1 for d in core if d > 0)
        coverage = covered / n * 100
        avg_depth = sum(core) / n

        if coverage >= min_cov and avg_depth >= min_depth:
            filtered_genes[gene] = {
                'coverage': coverage,
                'depth': avg_depth,
                'min_depth': min(core)
            }

    return filtered_genes


def join_genes_by_similarity(resistance_kmer, gene_length, filtered_genes,
                             dict_unique_hits, threshold=0.5):
    """Collapse near-identical alleles among the genes that passed the filter.

    If two genes share >= threshold of the SMALLER gene's reference k-mer
    set, they are treated as alleles of the same family and only one is
    kept. Tie-break order: coverage > avg depth > unique-hit count.
    """
    # Build a gene -> set of reference k-mers from the kmer index
    gene_kmers = {gene: set() for gene in gene_length}
    for kmer, entries in resistance_kmer.items():
        for gene, position, strand in entries:
            gene_kmers[gene].add(kmer)

    # Only compare genes that actually passed the filter 
    gene_list = list(filtered_genes.keys())
    removed = set()

    for i in range(len(gene_list)):
        for j in range(i + 1, len(gene_list)):
            gene_a, gene_b = gene_list[i], gene_list[j]

            if gene_a in removed or gene_b in removed:
                continue

            kmer_a, kmer_b = gene_kmers[gene_a], gene_kmers[gene_b]
            shared = len(kmer_a & kmer_b)   ### problem to have this here

            if shared == 0:
                continue
            
            # Problem here also
            smaller = kmer_a if len(kmer_a) <= len(kmer_b) else kmer_b
            similarity = shared / len(smaller)
            
            if similarity >= threshold:
                stat_a, stat_b = filtered_genes[gene_a], filtered_genes[gene_b]
                if stat_a['coverage'] != stat_b['coverage']:
                    better, worse = (gene_a, gene_b) if stat_a['coverage'] > stat_b['coverage'] else (gene_b, gene_a)
                elif stat_a['depth'] != stat_b['depth']:
                    better, worse = (gene_a, gene_b) if stat_a['depth'] > stat_b['depth'] else (gene_b, gene_a)
                else:
                    better, worse = (gene_a, gene_b) if dict_unique_hits[gene_a] >= dict_unique_hits[gene_b] else (gene_b, gene_a)
                removed.add(worse)

    final = {}
    for gene in filtered_genes:
        if gene not in removed:
            final[gene] = {
                'gene': gene,
                'coverage': filtered_genes[gene]['coverage'],
                'depth': filtered_genes[gene]['depth'],
                'min_depth': filtered_genes[gene]['min_depth'],
            }

    return final, removed


# ----------------------------
# MAIN RUNNING
# ----------------------------

if __name__ == '__main__':

    fasta_file = 'resistance_genes.fsa.txt'
    fastq_files = ['Unknown3_raw_reads_1.txt.gz', 'Unknown3_raw_reads_2.txt.gz']

    gene_length, gene_sequences = read_fasta_dict(fasta_file)
    resistance_kmer = create_dict_kmer(fasta_file)

    dict_depth, dict_unique_hits = match_reads_to_genes(
        fastq_files, resistance_kmer, gene_length)

    filtered_genes = calculate_coverage_depth(
        dict_depth,
        gene_length)

    final_genes, removed = join_genes_by_similarity(resistance_kmer, gene_length, filtered_genes,
                                dict_unique_hits, threshold=0.5)

    sorted_genes = sorted(final_genes.items(), key=lambda x: (-x[1]['coverage'], -x[1]['depth'], -x[1]['min_depth']))

    for gene, stats in sorted_genes:
        print(gene, "coverage:", round(stats['coverage'], 2), "depth:", round(stats['depth'], 2), "min_depth:", round(stats['min_depth'], 2))

catB4_1_EU935739 coverage: 100.0 depth: 144.62 min_depth: 38
fosA_3_ACWO01000079 coverage: 100.0 depth: 140.1 min_depth: 62
oqxA_1_EU370913 coverage: 100.0 depth: 132.53 min_depth: 94
oqxB_1_EU370913 coverage: 100.0 depth: 124.51 min_depth: 80
blaSHV-28_1_HM751101 coverage: 100.0 depth: 91.93 min_depth: 77
tet(A)_4_AJ517790 coverage: 100.0 depth: 70.49 min_depth: 44
aac(3)-IIa_1_CP023555.1 coverage: 100.0 depth: 56.91 min_depth: 43
aac(6')Ib-cr_1_DQ303918 coverage: 100.0 depth: 55.87 min_depth: 38
strB_1_M96392 coverage: 100.0 depth: 53.89 min_depth: 42
dfrA14_1_DQ388123 coverage: 100.0 depth: 52.41 min_depth: 31
blaCTX-M-15_23_DQ302097 coverage: 100.0 depth: 51.15 min_depth: 28
blaTEM-1B_1_JF910132 coverage: 100.0 depth: 50.46 min_depth: 38
strA_4_AF321551 coverage: 100.0 depth: 50.25 min_depth: 33
blaOXA-1_1_J02967 coverage: 100.0 depth: 49.52 min_depth: 36
sul2_2_GQ421466 coverage: 100.0 depth: 45.31 min_depth: 30
